# CryoSPH-CUDA quick start

GPU multiphase SPH for cryogenic propellant tanks in microgravity.  This notebook
builds the solver (CUDA backend on a GPU runtime, OpenMP backend otherwise),
runs a lateral thruster pulse on a half-full liquid-hydrogen tank and plots the
slosh force on the tank.

*Runtime → Change runtime type → GPU* for the CUDA backend.  Building takes a
few minutes.

In [ ]:
REPO_URL = "https://github.com/Semedo2002/Microgravity-Cryogenic-Multi-Phase-SPH-Solver"
!git clone --depth 1 $REPO_URL cryosph
!pip -q install cmake ninja pybind11 numpy matplotlib

In [ ]:
import shutil, subprocess
has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
cuda = "ON" if has_gpu else "OFF"
print("GPU runtime:", has_gpu)
!cmake -S cryosph -B build -G Ninja -DCMAKE_BUILD_TYPE=Release -DCRYOSPH_ENABLE_CUDA=$cuda \
      -DCMAKE_CUDA_ARCHITECTURES=native -DCRYOSPH_BUILD_TESTS=OFF -DCRYOSPH_BUILD_BENCHMARKS=OFF \
      -Dpybind11_DIR=$(python -m pybind11 --cmakedir) > configure.log && cmake --build build -j 4 | tail -2

In [ ]:
import sys
sys.path.insert(0, "build/python")
import numpy as np
import CryoSPH as cs
print(cs.__version__, "on", cs.device_name())

## A thruster pulse on a half-full LH2 tank

The tank sits in 1e-4 g.  A 0.2 s lateral pulse of 0.5 m/s² (x) is applied
through `ManeuverProfile`; the solver integrates in the accelerating tank frame
and reports the force the liquid exerts on the tank walls.

In [ ]:
lh2 = cs.FluidProperties.liquid_hydrogen()
cfg = cs.SolverConfig()
cfg.dx = 0.008
cfg.gravity = (0, 0, -9.81e-4)
cfg.contact_angle_deg = 5
cfg.thermal = False
cfg.phase_change = False
cfg.velocity_scale = 1.0  # c0 = 10 m/s: acoustic modes well above the slosh dynamics
solver = cs.Solver(cfg, lh2)
fill = cs.FillSpec(); fill.fill_fraction = 0.5
solver.build_tank(cs.CapsuleTank(radius=0.08, length=0.12), fill)
solver.relax(0.05)
print(solver.num_particles, "particles")

t = [0.0, 0.05, 0.25, 0.30]
acc = [[0, 0, 0], [0.5, 0, 0], [0.5, 0, 0], [0, 0, 0]]
solver.set_maneuver(cs.ManeuverProfile(t, np.array(acc)))
rows = []
while solver.time < 1.5:
    solver.run_until(solver.time + 0.01)
    d = solver.diagnostics()
    rows.append((d.time, *d.slosh_force, d.frame_acceleration[0]))
rows = np.array(rows)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(2, 1, figsize=(8, 5), sharex=True, constrained_layout=True)
ax[0].plot(rows[:, 0], rows[:, 4], color="#d9480f"); ax[0].set_ylabel("tank accel. x [m/s²]")
ax[1].plot(rows[:, 0], rows[:, 1], color="#2a6fdb", label="F_x")
ax[1].plot(rows[:, 0], rows[:, 2], color="#7a8699", label="F_y")
ax[1].set_ylabel("slosh force on tank [N]"); ax[1].set_xlabel("time [s]"); ax[1].legend(frameon=False)
plt.show()

Next steps: `examples/` contains boil-off, reorientation, high-evaporation and
manoeuvre drivers; `validation/spheric_test2` compares the solver with the MARIN
dam-break experiment; `viz/index.html` plays back the JSON frames that
`solver.write_json(...)` writes.